# Novel View Synthesis with Neural Fields (NeRF)

Compare neural field backbones on a NeRF synthetic (Blender) scene, under
FINER's protocol: [torch-ngp](https://github.com/ashawkey/torch-ngp) at
`--downscale 4 --trainskip 4 --bound 1 --scale 0.8 --dt_gamma 0 --iter 37500`,
i.e. 25 training views at 200x200 scored on all 200 test views. Every backbone
shares the same radiance-field heads and renderer, so the spatial
representation is the only variable.

## Setup

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display


import neurofield as nf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backend = "nerfacc" if nf.nerf.is_nerfacc_available() else "pytorch"
print(f"Device: {device}, renderer backend: {backend}")

## Load Data

In [ ]:
SCENE = "lego"
DATA_PATH = f"../data/nerf/blender/{SCENE}"
DOWNSAMPLE = 4  # 200x200 (torch-ngp --downscale)
TRAIN_SKIP = 4  # 25 of the 100 train views (torch-ngp --trainskip)


train_dataset = nf.nerf.BlenderDataset(
    DATA_PATH, split="train", downsample=DOWNSAMPLE, skip=TRAIN_SKIP
)
val_dataset = nf.nerf.BlenderDataset(DATA_PATH, split="val", downsample=DOWNSAMPLE)
test_dataset = nf.nerf.BlenderDataset(DATA_PATH, split="test", downsample=DOWNSAMPLE)

print(
    f"train: {len(train_dataset)}, val: {len(val_dataset)}, "
    f"test: {len(test_dataset)} views at "
    f"{train_dataset.height} x {train_dataset.width}"
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, idx in zip(axes, np.linspace(0, len(train_dataset) - 1, 4, dtype=int)):
    ax.imshow(train_dataset.composite(train_dataset.images[idx]).cpu())
    ax.set_title(f"View {idx}")
    ax.axis("off")
plt.show()

## Model Configurations

Every neural field has about 76k parameters, including the shared color network,
so results compare architectures at equal capacity. MFN and Instant-NGP follow
their official code; `# deviation` comments mark the changes needed under this
protocol.

In [ ]:
# Shared configuration across all models
GEOMETRY_FEATURES = 15
COLOR_NET = ("mlp", {"hidden_features": 64, "hidden_layers": 2})

# Model-specific configurations
CONFIGS = [
    {
        "name": "RFF",
        "density_net": (
            nf.RFF,
            {
                "hidden_features": 180,
                "hidden_layers": 2,
                "num_frequencies": 90,
                "sigma": 8.0,
            },
        ),
        "lr": 1e-2,
    },
    {
        "name": "PE-MLP",
        "density_net": (
            nf.PEMLP,
            {"hidden_features": 180, "hidden_layers": 3, "num_frequencies": 10},
        ),
        "lr": 1e-2,
    },
    {
        "name": "MFN",  # GaborNet (Fathony et al., ICLR 2021)
        "density_net": (
            nf.MFN,
            {
                "hidden_features": 176,
                "hidden_layers": 3,
                # deviation: the official input_scale 256 and alpha 6 are
                # 2.7 dB worse on the validation views.
                "input_scale": 128.0,
                "alpha": 2.0,
                "beta": 1.0,
                "weight_scale": 1.0,
            },
        ),
        "lr": 1e-2,
    },
    {
        "name": "SIREN",
        "density_net": (
            nf.SIREN,
            {"hidden_features": 180, "hidden_layers": 3, "omega": 30.0},
        ),
        "lr": 1e-3,
    },
    {
        "name": "Gauss",
        "density_net": (
            nf.Gauss,
            {"hidden_features": 180, "hidden_layers": 3, "scale": 30.0},
        ),
        "lr": 1e-3,
    },
    {
        "name": "WIRE",
        "density_net": (
            nf.RealWIRE,
            {"hidden_features": 180, "hidden_layers": 3, "omega": 20.0, "scale": 10.0},
        ),
        "lr": 3e-3,
    },
    {
        "name": "FINER",
        "density_net": (
            nf.FINER,
            {"hidden_features": 180, "hidden_layers": 3, "omega": 30.0},
        ),
        "lr": 1e-3,
    },
    {
        "name": "Instant-NGP",  # instant-ngp density network, paper hash grid
        "density_net": (
            nf.InstantNGP,
            {
                "num_levels": 16,
                "features_per_level": 2,
                "log2_hashmap_size": 11,
                "base_resolution": 16,
                "max_resolution": 2048,
                "hidden_features": 64,
                "hidden_layers": 1,
            },
        ),
        "lr": 1e-2,
    },
    {
        "name": "TensoRF-CP",
        "density_net": (
            nf.TensoRF,
            {
                "rank": 56,
                "resolution": 256,
                "mode": "cp",
                "hidden_features": 128,
                "hidden_layers": 2,
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "TensoRF-VM",
        "density_net": (
            nf.TensoRF,
            {
                "rank": 14,
                "resolution": 32,
                "mode": "vm",
                "hidden_features": 128,
                "hidden_layers": 2,
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "GA-Planes",
        "density_net": (
            nf.GAPlanes,
            {
                "resolution": (128, 32, 4),
                "features": 12,
                "hidden_features": 128,
                "hidden_layers": 2,
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "FUTON-cosine",
        "density_net": (
            nf.FUTON,
            {
                "basis": ("cosine", {"num_components": 128}),
                "combiner": ("cp", {"rank": 128}),
                "decoder": ("mlp", {"hidden_layers": 1}),
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "FUTON-sinc",
        "density_net": (
            nf.FUTON,
            {
                "basis": ("sinc", {"num_components": 128}),
                "combiner": ("cp", {"rank": 128}),
                "decoder": ("mlp", {"hidden_layers": 1}),
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "FUTON-lanczos",
        "density_net": (
            nf.FUTON,
            {
                "basis": ("lanczos", {"num_components": 128, "radius": 3}),
                "combiner": ("cp", {"rank": 128}),
                "decoder": ("mlp", {"hidden_layers": 1}),
            },
        ),
        "lr": 3e-2,
    },


## Train Models

In [ ]:
NUM_STEPS = 37_500  # FINER protocol (torch-ngp --iter); 5_000 for a quick pass
NUM_RAYS = 4_096  # torch-ngp --num_rays
MAX_SAMPLES_PER_RAY = 1_024  # torch-ngp --max_steps
EVAL_INTERVAL = 2_500
EVAL_VIEWS = list(range(0, len(val_dataset), 25))  # 4 val views for the curve
SEED = 0


def run(configs, **overrides):
    """Train every config on the shared protocol."""
    results = []
    for config in configs:
        torch.manual_seed(SEED)

        field = nf.nerf.RadianceField(
            density_net=config["density_net"],
            color_net=COLOR_NET,
            geometry_features=GEOMETRY_FEATURES,
            direction_encoding="sh",
            aabb=train_dataset.aabb,
        )
        renderer = nf.nerf.create_renderer(
            backend=backend,
            aabb=train_dataset.aabb,
            near=train_dataset.near,
            far=train_dataset.far,
        )
        num_params = nf.count_parameters(field)

        print(f"\n--- {config['name']} ({num_params:,} params) ---")
        res = nf.nerf.train(
            field,
            renderer,
            train_dataset,
            eval_dataset=val_dataset,
            eval_indices=EVAL_VIEWS,
            device=device,
            **{
                "num_steps": NUM_STEPS,
                "num_rays": NUM_RAYS,
                "max_samples_per_ray": MAX_SAMPLES_PER_RAY,
                "lr": config["lr"],
                "eval_interval": EVAL_INTERVAL,
                "log_interval": 100,
                "seed": SEED,
                **overrides,
            },
        )

        res["config"]["model"] = config["name"]
        res["field"], res["renderer"] = field, renderer
        results.append(res)
        torch.cuda.empty_cache()
    return results


results = run(CONFIGS)

## Collect Results

In [ ]:
records = []
for res in results:
    for entry in res["history"]:
        if "eval" not in entry:
            continue

        records.append(
            {
                "Model": res["config"]["model"],
                "# Params (k)": res["config"]["num_params"] / 1000,
                "Iteration": entry["step"],
                "Time (s)": entry["elapsed"],
                "PSNR (dB)": entry["eval"]["psnr"],
                "SSIM": entry["eval"]["ssim"],
                "LPIPS": entry["eval"]["lpips"],
            }
        )


records = pd.DataFrame(records)

## Convergence Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, x in zip(axes, ["Iteration", "Time (s)"]):
    sns.lineplot(
        data=records,
        x=x,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        markers=True,
        ax=ax,
    )
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
TEST_INDICES = range(len(test_dataset))  # all 200; range(0, 200, 10) to skim


def summarize(results):
    """Score every result on the held-out test split."""
    for res in results:
        res["eval"] = nf.nerf.evaluate(
            res["field"],
            res["renderer"],
            test_dataset,
            indices=TEST_INDICES,
        )

    return pd.DataFrame(
        [
            {
                "Model": res["config"]["model"],
                "# Params (k)": res["config"]["num_params"] / 1000,
                "Time (s)": res["history"][-1]["elapsed"],
                "Speed (views/s)": 1.0 / res["eval"]["mean"]["duration"],
                "PSNR (dB)": res["eval"]["mean"]["psnr"],
                "SSIM": res["eval"]["mean"]["ssim"],
                "LPIPS": res["eval"]["mean"]["lpips"],
            }
            for res in results
        ]
    ).sort_values("PSNR (dB)", ascending=False)


summary = summarize(results).reset_index(drop=True)
display(summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, metric in zip(axes, ["PSNR (dB)", "SSIM", "LPIPS", "# Params (k)"]):
    ordered = summary.sort_values(metric, ascending=metric != "LPIPS")
    ax.barh(ordered["Model"], ordered[metric])
    ax.set_xlabel(metric)
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## Visual Comparison

In [ ]:
VIEW_IDS = [0, 55]

psnr_lookup = summary.set_index("Model")["PSNR (dB)"]
sorted_results = sorted(
    results, key=lambda r: psnr_lookup[r["config"]["model"]], reverse=True
)

for view in VIEW_IDS:
    batch = test_dataset[view]
    target = test_dataset.composite(batch["rgba"])
    num_cols = 1 + len(sorted_results)
    fig, axes = plt.subplots(2, num_cols, figsize=(2.5 * num_cols, 6))
    fig.suptitle(f"{SCENE} - test view {view}")

    axes[0, 0].imshow(target)
    axes[0, 0].set_title("Ground truth")

    for col, res in enumerate(sorted_results, start=1):
        rendered = nf.nerf.render_image(
            res["field"],
            res["renderer"],
            batch["rays_o"],
            batch["rays_d"],
            background=test_dataset.background.to(device),
        )
        pred = rendered["rgb"].clamp(0, 1).cpu()
        name = res["config"]["model"]

        axes[0, col].imshow(pred)
        axes[0, col].set_title(f"\n{name}\nPSNR: {psnr_lookup[name]:.1f}dB")
        axes[1, col].imshow(
            (pred - target).abs().mean(-1), cmap="inferno", vmin=0, vmax=0.25
        )

    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Novel-View Orbit

In [ ]:
best = sorted_results[0]
print(f"Rendering orbit with {best['config']['model']}")

# The dataset's own camera elevation and radius, in its scaled frame.
poses = [
    nf.nerf.pose_spherical(theta, -30.0, test_dataset.radius)
    for theta in np.linspace(-180, 180, 60, endpoint=False)
]
frames = nf.nerf.render_views(best["field"], best["renderer"], test_dataset, poses)

with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = Path(tmpdir) / f"{SCENE}_orbit.gif"
    imageio.mimsave(gif_path, frames, fps=15, loop=0)
    display(IPyImage(filename=str(gif_path)))